In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [13]:
data = pd.read_csv('heart_disease_uci(1).csv')

# Drop 'id' column if exists
if 'id' in data.columns:
    data = data.drop(columns=['id'])

# Remove rows with missing values
data.dropna(inplace=True)

# Remove duplicate rows
data.drop_duplicates(inplace=True)

print("✅ After Data Cleaning:")
print(data.head())

✅ After Data Cleaning:
   age     sex    dataset               cp  trestbps   chol    fbs  \
0   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  
0       fixed defect    0  
1             normal    2  
2  reversable defect    1  
3             normal    0  
4             normal    0  


In [14]:
data

,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,68,Male,Cleveland,asymptomatic,144.0,193.0,True,normal,141.0,False,3.4,flat,2.0,reversable defect,2
300,57,Male,Cleveland,asymptomatic,130.0,131.0,False,normal,115.0,True,1.2,flat,1.0,reversable defect,3
301,57,Female,Cleveland,atypical angina,130.0,236.0,False,lv hypertrophy,174.0,False,0.0,flat,1.0,normal,1
508,47,Male,Hungary,asymptomatic,150.0,226.0,False,normal,98.0,True,1.5,flat,0.0,reversable defect,1


In [15]:
df1=data[['age','sex']]
df1

,age,sex
0,63,Male
1,67,Male
2,67,Male
3,37,Male
4,41,Female
...,...,...
299,68,Male
300,57,Male
301,57,Female
508,47,Male


In [16]:
df2=data[['trestbps','chol']]
df2

,trestbps,chol
0,145.0,233.0
1,160.0,286.0
2,120.0,229.0
3,130.0,250.0
4,130.0,204.0
...,...,...
299,144.0,193.0
300,130.0,131.0
301,130.0,236.0
508,150.0,226.0


In [21]:
merged_data=pd.concat([df1,df2],axis=1)
merged_data

,age,sex,trestbps,chol
0,63,Male,145.0,233.0
1,67,Male,160.0,286.0
2,67,Male,120.0,229.0
3,37,Male,130.0,250.0
4,41,Female,130.0,204.0
...,...,...,...,...
299,68,Male,144.0,193.0
300,57,Male,130.0,131.0
301,57,Female,130.0,236.0
508,47,Male,150.0,226.0


In [17]:
for col in data.columns:
    if data[col].dtype == 'object':
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])

print("\n✅ After Data Transformation:")
print(data.head())


✅ After Data Transformation:
   age  sex  dataset  cp  trestbps   chol  fbs  restecg  thalch  exang  \
0   63    1        0   3     145.0  233.0    1        0   150.0      0   
1   67    1        0   0     160.0  286.0    0        0   108.0      1   
2   67    1        0   0     120.0  229.0    0        0   129.0      1   
3   37    1        0   2     130.0  250.0    0        1   187.0      0   
4   41    0        0   1     130.0  204.0    0        0   172.0      0   

   oldpeak  slope   ca  thal  num  
0      2.3      0  0.0     0    0  
1      1.5      1  3.0     1    2  
2      2.6      1  2.0     2    1  
3      3.5      0  0.0     1    0  
4      1.4      2  0.0     1    0  


In [18]:
def remove_outliers(df, column):
    low = df[column].quantile(0.01)
    high = df[column].quantile(0.99)
    return df[(df[column] >= low) & (df[column] <= high)]

# Apply outlier removal to all numeric columns except target
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    if col != 'num':  # Skip target column
        data = remove_outliers(data, col)

# Reset index after filtering
data.reset_index(drop=True, inplace=True)

print("\n✅ After Error Correcting (Outlier Removal):")
print(data.describe())


✅ After Error Correcting (Outlier Removal):
              age         sex  dataset          cp    trestbps        chol  \
count  270.000000  270.000000    270.0  270.000000  270.000000  270.000000   
mean    54.529630    0.688889      0.0    0.951852  131.403704  247.218519   
std      8.672321    0.463808      0.0    1.035385   16.685924   45.405734   
min     35.000000    0.000000      0.0    0.000000  100.000000  149.000000   
25%     48.000000    0.000000      0.0    0.000000  120.000000  212.250000   
50%     56.000000    1.000000      0.0    1.000000  130.000000  243.500000   
75%     61.000000    1.000000      0.0    2.000000  140.000000  275.000000   
max     71.000000    1.000000      0.0    3.000000  180.000000  407.000000   

              fbs     restecg      thalch       exang     oldpeak       slope  \
count  270.000000  270.000000  270.000000  270.000000  270.000000  270.000000   
mean     0.140741    0.507407  149.662963    0.322222    1.020741    1.414815   
std      

In [19]:
X = data.drop(columns=['num'])  # Features
y = data['num']                 # Target

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

# 1. Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

# 2. Decision Tree Classifier
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

# Evaluation
print("\n✅ === Logistic Regression Evaluation ===")
print("Accuracy:", accuracy_score(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

print("\n✅ === Decision Tree Classifier Evaluation ===")
print("Accuracy:", accuracy_score(y_test, dt_pred))
print(classification_report(y_test, dt_pred))



✅ === Logistic Regression Evaluation ===
Accuracy: 0.5740740740740741
              precision    recall  f1-score   support

           0       0.71      0.96      0.82        26
           1       0.43      0.19      0.26        16
           2       0.14      0.33      0.20         3
           3       0.40      0.33      0.36         6
           4       0.00      0.00      0.00         3

    accuracy                           0.57        54
   macro avg       0.34      0.36      0.33        54
weighted avg       0.52      0.57      0.52        54


✅ === Decision Tree Classifier Evaluation ===
Accuracy: 0.4444444444444444
              precision    recall  f1-score   support

           0       0.67      0.77      0.71        26
           1       0.30      0.19      0.23        16
           2       0.00      0.00      0.00         3
           3       0.14      0.17      0.15         6
           4       0.00      0.00      0.00         3

    accuracy                          

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre